In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
from IPython.display import display

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
url = "https://github.com/trending"
headers = {"User-Agent": "Mozilla/5.0"}

def scrape_github_trending():
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    rows = []

    for article in soup.select("article.Box-row"):
        title_tag = article.select_one("h2 a")
        if title_tag is None:
            continue

        repo_path = title_tag.get("href", "").strip()
        repo_name = "/".join(part for part in repo_path.split("/") if part)
        description_tag = article.select_one("p")
        language_tag = article.select_one('[itemprop="programmingLanguage"]')

        stars = 0
        stars_tag = article.select_one('a[href$="/stargazers"]')
        if stars_tag:
            star_text = stars_tag.get_text(" ", strip=True).replace(",", "")
            try:
                stars = int(star_text)
            except ValueError:
                stars = 0

        today_stars = 0
        for span in article.select("span"):
            text_value = span.get_text(" ", strip=True)
            if "stars today" in text_value.lower() or "stars this week" in text_value.lower():
                number = text_value.lower().split(" star")[0].replace(",", "").strip()
                try:
                    today_stars = int(number)
                except ValueError:
                    pass
                break

        rows.append({
            "repository": repo_name,
            "description": description_tag.get_text(" ", strip=True) if description_tag else "",
            "language": language_tag.get_text(strip=True) if language_tag else "Unknown",
            "stars": stars,
            "stars_today": today_stars,
            "url": "https://github.com" + repo_path
        })

    if not rows:
        raise ValueError("No repositories found. GitHub's page structure may have changed or access may be blocked.")

    return pd.DataFrame(rows)

try:
    df = scrape_github_trending()
    print(f"Successfully scraped {len(df)} repositories from GitHub Trending.")
except Exception as error:
    print("Live scraping did not work:", error)
    print("Creating demo data so you can still test the dashboard. This is NOT live scraped data.")
    df = pd.DataFrame([
        {"repository":"owner/project-alpha","description":"Example data tools","language":"Python","stars":12500,"stars_today":320,"url":"https://github.com"},
        {"repository":"owner/project-beta","description":"A web framework example","language":"JavaScript","stars":9800,"stars_today":210,"url":"https://github.com"},
        {"repository":"owner/project-gamma","description":"Machine learning utilities","language":"Python","stars":8700,"stars_today":185,"url":"https://github.com"},
        {"repository":"owner/project-delta","description":"Command line tools","language":"Go","stars":6200,"stars_today":140,"url":"https://github.com"},
        {"repository":"owner/project-epsilon","description":"UI component examples","language":"TypeScript","stars":5100,"stars_today":95,"url":"https://github.com"},
        {"repository":"owner/project-zeta","description":"Data visualization examples","language":"R","stars":4300,"stars_today":70,"url":"https://github.com"},
        {"repository":"owner/project-eta","description":"Automation examples","language":"Python","stars":3900,"stars_today":60,"url":"https://github.com"},
        {"repository":"owner/project-theta","description":"Backend service examples","language":"Java","stars":3100,"stars_today":45,"url":"https://github.com"},
        {"repository":"owner/project-iota","description":"CSS toolkit example","language":"CSS","stars":2400,"stars_today":35,"url":"https://github.com"},
        {"repository":"owner/project-kappa","description":"Rust learning project","language":"Rust","stars":1800,"stars_today":25,"url":"https://github.com"}
    ])

display(df.head())

Successfully scraped 8 repositories from GitHub Trending.


,repository,description,language,stars,stars_today,url
0,morluto/rea,"Reverse engineer anything with agents, from ap...",TypeScript,93102,25585,https://github.com/morluto/rea
1,boykopovar/AnyPS5,Tool for automatic PS5 executables porting to ...,C++,28915,3977,https://github.com/boykopovar/AnyPS5
2,storytold/artcraft,ArtCraft is an intentional crafting engine for...,Rust,16609,2956,https://github.com/storytold/artcraft
3,cathrynlavery/diagram-design,"Editorial diagram design for Claude Code, Code...",HTML,49861,1083,https://github.com/cathrynlavery/diagram-design
4,bilawalsidhu/gods-eye-view,"A spy satellite simulator in your browser, exc...",JavaScript,51045,872,https://github.com/bilawalsidhu/gods-eye-view


In [3]:
df["language"] = df["language"].fillna("Unknown").replace("", "Unknown")
df["stars"] = pd.to_numeric(df["stars"], errors="coerce").fillna(0).astype(int)
df["stars_today"] = pd.to_numeric(df["stars_today"], errors="coerce").fillna(0).astype(int)
df = df.drop_duplicates(subset=["repository"]).reset_index(drop=True)

csv_file = "github_trending_repositories.csv"
df.to_csv(csv_file, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to {csv_file}")
display(df)

Saved 8 rows to github_trending_repositories.csv


,repository,description,language,stars,stars_today,url
0,morluto/rea,"Reverse engineer anything with agents, from ap...",TypeScript,93102,25585,https://github.com/morluto/rea
1,boykopovar/AnyPS5,Tool for automatic PS5 executables porting to ...,C++,28915,3977,https://github.com/boykopovar/AnyPS5
2,storytold/artcraft,ArtCraft is an intentional crafting engine for...,Rust,16609,2956,https://github.com/storytold/artcraft
3,cathrynlavery/diagram-design,"Editorial diagram design for Claude Code, Code...",HTML,49861,1083,https://github.com/cathrynlavery/diagram-design
4,bilawalsidhu/gods-eye-view,"A spy satellite simulator in your browser, exc...",JavaScript,51045,872,https://github.com/bilawalsidhu/gods-eye-view
5,pytorch/pytorch,Tensors and Dynamic neural networks in Python ...,Python,104345,314,https://github.com/pytorch/pytorch
6,lyogavin/airllm,AirLLM 70B inference with single 4GB GPU,Jupyter Notebook,36235,541,https://github.com/lyogavin/airllm
7,chrislgarry/Apollo-11,Original Apollo 11 Guidance Computer (AGC) sou...,Assembly,73990,496,https://github.com/chrislgarry/Apollo-11


In [4]:
app = Dash(__name__)

def make_kpi(title, value):
    return html.Div(
        [
            html.Div(title, style={"fontSize": "14px", "color": "#555"}),
            html.H2(value, style={"margin": "8px 0", "color": "#172554"})
        ],
        style={
            "backgroundColor": "#f1f5f9",
            "padding": "18px",
            "borderRadius": "12px",
            "flex": "1",
            "minWidth": "150px",
            "boxShadow": "0 1px 4px rgba(0,0,0,0.08)"
        }
    )

languages = sorted(df["language"].dropna().unique().tolist())
dropdown_options = [{"label": "All languages", "value": "All"}] + [
    {"label": lang, "value": lang} for lang in languages
]

app.layout = html.Div(
    [
        html.H1("GitHub Trending Repositories Dashboard",
                style={"textAlign": "center", "color": "#172554"}),
        html.P("Explore trending repositories and compare stars by programming language.",
               style={"textAlign": "center", "color": "#475569"}),
        html.Label("Filter by programming language:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="language-filter",
            options=dropdown_options,
            value="All",
            clearable=False,
            style={"marginBottom": "20px"}
        ),
        html.Div(id="kpi-cards", style={"display": "flex", "gap": "12px", "flexWrap": "wrap"}),
        html.Div([
            dcc.Graph(id="stars-chart"),
            dcc.Graph(id="language-chart")
        ], style={"display": "grid", "gridTemplateColumns": "repeat(auto-fit, minmax(320px, 1fr))", "gap": "15px"}),
        html.H3("Repository Details"),
        html.Div(id="repository-table")
    ],
    style={"fontFamily": "Arial, sans-serif", "maxWidth": "1200px", "margin": "auto", "padding": "24px"}
)

@app.callback(
    Output("kpi-cards", "children"),
    Output("stars-chart", "figure"),
    Output("language-chart", "figure"),
    Output("repository-table", "children"),
    Input("language-filter", "value")
)
def update_dashboard(selected_language):
    filtered = df if selected_language == "All" else df[df["language"] == selected_language]
    repo_count = len(filtered)
    total_stars = int(filtered["stars"].sum()) if repo_count else 0
    average_stars = int(filtered["stars"].mean()) if repo_count else 0
    total_today = int(filtered["stars_today"].sum()) if repo_count else 0

    cards = [
        make_kpi("Repositories", f"{repo_count:,}"),
        make_kpi("Total Stars", f"{total_stars:,}"),
        make_kpi("Average Stars", f"{average_stars:,}"),
        make_kpi("Stars Today (reported)", f"{total_today:,}")
    ]

    stars_fig = px.bar(
        filtered.sort_values("stars", ascending=False).head(10),
        x="repository", y="stars", title="Top Repositories by Stars",
        labels={"repository": "Repository", "stars": "Stars"},
        hover_data=["language", "stars_today"]
    )
    stars_fig.update_layout(xaxis_tickangle=-35)

    language_counts = filtered["language"].value_counts().rename_axis("language").reset_index(name="repositories")
    language_fig = px.pie(
        language_counts, names="language", values="repositories",
        title="Programming Language Distribution"
    )

    if repo_count:
        table_rows = [
            html.Tr([html.Th("Repository"), html.Th("Language"), html.Th("Stars"), html.Th("URL")])
        ]
        for _, row in filtered.iterrows():
            table_rows.append(html.Tr([
                html.Td(row["repository"]),
                html.Td(row["language"]),
                html.Td(f'{row["stars"]:,}'),
                html.Td(html.A("Open", href=row["url"], target="_blank"))
            ]))
        table = html.Table(table_rows, style={"width": "100%", "textAlign": "left", "borderSpacing": "8px"})
    else:
        table = html.P("No repositories match this filter.")

    return cards, stars_fig, language_fig, table

print("Dashboard app is ready.")

Dashboard app is ready.


In [6]:

app.run(debug=False, port=8050,jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/
